# Ungraded Lab: Fully Convolutional Neural Networks for Image Segmentation (PyTorch)

This notebook illustrates how to build a Fully Convolutional Neural Network for semantic image segmentation.

You will train the model on a [custom dataset](https://drive.google.com/file/d/0B0d9ZiqAgFkiOHR1NTJhWVJMNEU/view?usp=sharing) prepared by [divamgupta](https://github.com/divamgupta/image-segmentation-keras). This contains video frames from a moving vehicle and is a subsample of the [CamVid](http://mi.eng.cam.ac.uk/research/projects/VideoRec/CamVid/) dataset.

You will be using a pretrained VGG-16 network for the feature extraction path, then followed by an FCN-8 network for upsampling and generating the predictions. The output will be a label map (i.e. segmentation mask) with predictions for 12 classes. Let's begin!

> This notebook is a PyTorch port of the original TensorFlow/Keras lab. The `tf.data` pipeline becomes a `Dataset`/`DataLoader`, the VGG-16 encoder and FCN-8 decoder become `nn.Module`s (with the pretrained VGG-16 weights copied from torchvision), and `model.fit` becomes an explicit training loop.

## Imports

In [ ]:
import os
import zipfile
import platform
import urllib.request
import PIL.Image, PIL.ImageFont, PIL.ImageDraw
import numpy as np

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vgg16, VGG16_Weights
from torchinfo import summary
from matplotlib import pyplot as plt
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

## Download the Dataset

We hosted the dataset in a Google bucket so you will need to download it first and unzip to a local directory.

In [ ]:
# download the dataset (zipped file)
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/fcnn-dataset.zip"):
    urllib.request.urlretrieve("https://storage.googleapis.com/learning-datasets/fcnn-dataset.zip", "data/fcnn-dataset.zip")

You can extract the downloaded zip files with this code:

In [ ]:
# extract the downloaded dataset to a local directory: data/fcnn
local_zip = 'data/fcnn-dataset.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('data/fcnn')
zip_ref.close()

The dataset you just downloaded contains folders for images and annotations. The *images* contain the video frames while the *annotations* contain the pixel-wise label maps. Each label map has the shape `(height, width , 1)` with each point in this space denoting the corresponding pixel's class. Classes are in the range `[0, 11]` (i.e. 12 classes) and the pixel labels correspond to these classes:

| Value  | Class Name    |
| -------| -------------|
| 0      | sky |
| 1      | building      |
| 2      | column/pole      |
| 3      | road |
| 4      | side walk     |
| 5      | vegetation      |
| 6      | traffic light |
| 7      | fence      |
| 8      | vehicle     |
| 9      | pedestrian |
| 10      | byciclist      |
| 11      | void      |

For example, if a pixel is part of a road, then that point will be labeled `3` in the label map. Run the cell below to create a list containing the class names:
- Note: bicyclist is mispelled as 'byciclist' in the dataset.  We won't handle data cleaning in this example, but you can inspect and clean the data if you want to use this as a starting point for a personal project.

In [ ]:
# pixel labels in the video frames
class_names = ['sky', 'building','column/pole', 'road', 'side walk', 'vegetation', 'traffic light', 'fence', 'vehicle', 'pedestrian', 'byciclist', 'void']

## Load and Prepare the Dataset

Next, you will load and prepare the train and validation sets for training. There are some preprocessing steps needed before the data is fed to the model. These include:

* resizing the height and width of the input images and label maps (224 x 224px by default)
* normalizing the input images' pixel values to fall in the range `[-1, 1]`
* converting the label maps to a `(height, width)` tensor of integer class ids.

In the Keras version the label maps were reshaped to one-hot `(height, width, 12)` tensors because `categorical_crossentropy` needs them. PyTorch's `nn.CrossEntropyLoss` takes the integer class ids directly, so that step is not needed here: the annotation stays a `(height, width)` map where, for example, a road pixel simply holds the value `3`.

The following function will do the preprocessing steps mentioned above.

In [ ]:
def map_filename_to_image_and_mask(t_filename, a_filename, height=224, width=224):
  '''
  Preprocesses the dataset by:
    * resizing the input image and label maps
    * normalizing the input image pixels
    * converting the label map to a (height, width) tensor of class ids

  Args:
    t_filename (string) -- path to the raw input image
    a_filename (string) -- path to the raw annotation (label map) file
    height (int) -- height in pixels to resize to
    width (int) -- width in pixels to resize to

  Returns:
    image (tensor) -- preprocessed image, shape (3, height, width)
    annotation (tensor) -- preprocessed annotation, shape (height, width)
  '''

  # Read the image and mask files
  image = PIL.Image.open(t_filename).convert("RGB")
  annotation = PIL.Image.open(a_filename)

  # Resize image and segmentation mask (nearest neighbour for the mask so that no new class ids are invented)
  image = image.resize((width, height), PIL.Image.BILINEAR)
  annotation = annotation.resize((width, height), PIL.Image.NEAREST)

  # Convert to tensors: image -> (3, height, width) float, annotation -> (height, width) int64
  image = torch.from_numpy(np.array(image, dtype=np.float32)).permute(2, 0, 1)
  annotation = torch.from_numpy(np.array(annotation)).long()

  # Normalize pixels in the input image
  image = image/127.5
  image -= 1

  return image, annotation

The dataset also already has separate folders for train and test sets. As described earlier, these sets will have two folders: one corresponding to the images, and the other containing the annotations.

In [ ]:
# show folders inside the dataset you downloaded
print(os.listdir('data/fcnn/dataset1'))

You will use the following functions to create the PyTorch datasets from the images in these folders. Notice that the images are preprocessed using the `map_filename_to_image_and_mask()` function you defined earlier every time an item is fetched from the `Dataset`, and the `DataLoader` takes care of the shuffling and batching in `get_training_dataset()` and `get_validation_dataset()`.

In [ ]:
# Utilities for preparing the datasets

BATCH_SIZE = 64

# DataLoader worker processes on macOS are started with "spawn", which cannot see classes defined
# inside a notebook (such as the Dataset below). "fork" works fine for the image decoding the workers do.
MP_CONTEXT = "fork" if platform.system() == "Darwin" else None


def get_dataset_slice_paths(image_dir, label_map_dir):
  '''
  generates the lists of image and label map paths

  Args:
    image_dir (string) -- path to the input images directory
    label_map_dir (string) -- path to the label map directory

  Returns:
    image_paths (list of strings) -- paths to each image file
    label_map_paths (list of strings) -- paths to each label map
  '''
  # sorted so that image i and label map i refer to the same frame
  image_file_list = sorted(os.listdir(image_dir))
  label_map_file_list = sorted(os.listdir(label_map_dir))
  image_paths = [os.path.join(image_dir, fname) for fname in image_file_list]
  label_map_paths = [os.path.join(label_map_dir, fname) for fname in label_map_file_list]

  return image_paths, label_map_paths


class SegmentationDataset(Dataset):
  '''pairs of (preprocessed image, label map)'''

  def __init__(self, image_paths, label_map_paths):
    '''
    Stores the paired image and label-map paths this split serves.

    Args:
      image_paths (list of str) -- paths to the input images
      label_map_paths (list of str) -- matching paths to the label maps
    '''
    self.image_paths = image_paths
    self.label_map_paths = label_map_paths

  def __len__(self):
    '''
    Reports how many pairs this split holds.

    Returns:
      int -- number of image and label-map pairs
    '''
    return len(self.image_paths)

  def __getitem__(self, idx):
    '''
    Loads and preprocesses pair `idx`.

    Args:
      idx (int) -- index of the pair to fetch

    Returns:
      (tensor, tensor) -- preprocessed image (3, 224, 224) and label map (224, 224)
    '''
    return map_filename_to_image_and_mask(self.image_paths[idx], self.label_map_paths[idx])


def get_training_dataset(image_paths, label_map_paths):
  '''
  Prepares shuffled batches of the training set.

  Args:
    image_paths (list of strings) -- paths to each image file in the train set
    label_map_paths (list of strings) -- paths to each label map in the train set

  Returns:
    DataLoader containing the preprocessed train set
  '''
  training_dataset = SegmentationDataset(image_paths, label_map_paths)
  training_loader = DataLoader(training_dataset, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=4, multiprocessing_context=MP_CONTEXT, persistent_workers=True)

  return training_loader


def get_validation_dataset(image_paths, label_map_paths):
  '''
  Prepares batches of the validation set.

  Args:
    image_paths (list of strings) -- paths to each image file in the val set
    label_map_paths (list of strings) -- paths to each label map in the val set

  Returns:
    DataLoader containing the preprocessed validation set
  '''
  validation_dataset = SegmentationDataset(image_paths, label_map_paths)
  validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE,
                                 num_workers=4, multiprocessing_context=MP_CONTEXT, persistent_workers=True)

  return validation_loader

You can now generate the training and validation sets by running the cell below.

In [ ]:
# get the paths to the images
training_image_paths, training_label_map_paths = get_dataset_slice_paths('data/fcnn/dataset1/images_prepped_train/','data/fcnn/dataset1/annotations_prepped_train/')
validation_image_paths, validation_label_map_paths = get_dataset_slice_paths('data/fcnn/dataset1/images_prepped_test/','data/fcnn/dataset1/annotations_prepped_test/')

# generate the train and val sets
training_dataset = get_training_dataset(training_image_paths, training_label_map_paths)
validation_dataset = get_validation_dataset(validation_image_paths, validation_label_map_paths)

## Let's Take a Look at the Dataset

You will also need utilities to help visualize the dataset and the model predictions later. First, you need to assign a color mapping to the classes in the label maps. Since our dataset has 12 classes, you need to have a list of 12 colors. We can use the [color_palette()](https://seaborn.pydata.org/generated/seaborn.color_palette.html) from Seaborn to generate this.

In [ ]:
# generate a list that contains one color for each class
colors = sns.color_palette(None, len(class_names))

# print class name - normalized RGB tuple pairs
# the tuple values will be multiplied by 255 in the helper functions later
# to convert to the (0,0,0) to (255,255,255) RGB values you might be familiar with
for class_name, color in zip(class_names, colors):
  print(f'{class_name} -- {color}')

In [ ]:
# Visualization Utilities

def to_numpy_image(image):
  '''
  Converts a normalized image tensor back to something matplotlib can show.

  Args:
    image (tensor) -- image in the range [-1, 1], shape (3, height, width)

  Returns:
    array -- uint8 image of shape (height, width, 3)
  '''
  image = np.asarray(image)
  if image.ndim == 3 and image.shape[0] == 3:
    image = image.transpose(1, 2, 0)
  image = image + 1
  image = image * 127.5
  return np.uint8(image)


def fuse_with_pil(images):
  '''
  Creates a blank image and pastes input images

  Args:
    images (list of numpy arrays) - numpy array representations of the images to paste

  Returns:
    PIL Image object containing the images
  '''

  widths = (image.shape[1] for image in images)
  heights = (image.shape[0] for image in images)
  total_width = sum(widths)
  max_height = max(heights)

  new_im = PIL.Image.new('RGB', (total_width, max_height))

  x_offset = 0
  for im in images:
    pil_image = PIL.Image.fromarray(np.uint8(im))
    new_im.paste(pil_image, (x_offset,0))
    x_offset += im.shape[1]

  return new_im


def give_color_to_annotation(annotation):
  '''
  Converts a 2-D annotation to a numpy array with shape (height, width, 3) where
  the third axis represents the color channel. The label values are multiplied by
  255 and placed in this axis to give color to the annotation

  Args:
    annotation (numpy array) - label map array

  Returns:
    the annotation array with an additional color channel/axis
  '''
  annotation = np.asarray(annotation)
  seg_img = np.zeros( (annotation.shape[0],annotation.shape[1], 3) ).astype('float')

  for c in range(12):
    segc = (annotation == c)
    seg_img[:,:,0] += segc*( colors[c][0] * 255.0)
    seg_img[:,:,1] += segc*( colors[c][1] * 255.0)
    seg_img[:,:,2] += segc*( colors[c][2] * 255.0)

  return seg_img


def show_predictions(image, labelmaps, titles, iou_list, dice_score_list):
  '''
  Displays the images with the ground truth and predicted label maps

  Args:
    image (tensor or numpy array) -- the input image, shape (3, height, width), range [-1, 1]
    labelmaps (list of arrays) -- contains the predicted and ground truth label maps
    titles (list of strings) -- display headings for the images to be displayed
    iou_list (list of floats) -- the IOU values for each class
    dice_score_list (list of floats) -- the Dice Score for each vlass
  '''

  true_img = give_color_to_annotation(labelmaps[1])
  pred_img = give_color_to_annotation(labelmaps[0])

  image = to_numpy_image(image)
  images = np.uint8([image, pred_img, true_img])

  metrics_by_id = [(idx, iou, dice_score) for idx, (iou, dice_score) in enumerate(zip(iou_list, dice_score_list)) if iou > 0.0]
  metrics_by_id.sort(key=lambda tup: tup[1], reverse=True)  # sorts in place

  display_string_list = ["{}: IOU: {} Dice Score: {}".format(class_names[idx], iou, dice_score) for idx, iou, dice_score in metrics_by_id]
  display_string = "\n\n".join(display_string_list)

  plt.figure(figsize=(15, 4))

  for idx, im in enumerate(images):
    plt.subplot(1, 3, idx+1)
    if idx == 1:
      plt.xlabel(display_string)
    plt.xticks([])
    plt.yticks([])
    plt.title(titles[idx], fontsize=12)
    plt.imshow(im)


def show_annotation_and_image(image, annotation):
  '''
  Displays the image and its annotation side by side

  Args:
    image (tensor) -- the input image, shape (3, height, width)
    annotation (tensor) -- the label map, shape (height, width)
  '''
  seg_img = give_color_to_annotation(annotation)

  image = to_numpy_image(image)
  images = [image, seg_img]

  fused_img = fuse_with_pil(images)
  plt.imshow(fused_img)


def list_show_annotation(dataloader):
  '''
  Displays images and its annotations side by side

  Args:
    dataloader (DataLoader) - batches of images and annotations
  '''

  ds = dataloader.dataset

  plt.figure(figsize=(25, 15))
  plt.title("Images And Annotations")
  plt.subplots_adjust(bottom=0.1, top=0.9, hspace=0.05)

  # we set the number of image-annotation pairs to 9
  # feel free to make this a function parameter if you want
  for idx, i in enumerate(np.random.choice(len(ds), size=9, replace=False)):
    image, annotation = ds[i]
    plt.subplot(3, 3, idx + 1)
    plt.yticks([])
    plt.xticks([])
    show_annotation_and_image(image.numpy(), annotation.numpy())

Please run the cells below to see sample images from the train and validation sets. You will see the image and the label maps side side by side.

In [ ]:
list_show_annotation(training_dataset)

In [ ]:
list_show_annotation(validation_dataset)

## Define the Model

You will now build the model and prepare it for training. AS mentioned earlier, this will use a VGG-16 network for the encoder and FCN-8 for the decoder. This is the diagram as shown in class:

<img src='https://drive.google.com/uc?export=view&id=1lrqB4YegV8jXWNfyYAaeuFlwXIc54aRP' alt='fcn-8'>

For this exercise, you will notice a slight difference from the lecture because the dataset images are 224x224 instead of 32x32. You'll see how this is handled in the next cells as you build the encoder.

### Define Pooling Block of VGG

As you saw in Course 1 of this specialization, VGG networks have repeating blocks so to make the code neat, it's best to create a function to encapsulate this process. Each block has convolutional layers followed by a max pooling layer which downsamples the image.

In PyTorch, convolution layers need to know their number of input channels, so the function takes `in_channels` in addition to the number of `filters`. `padding='same'` keeps the spatial size unchanged, just like in Keras.

In [ ]:
def block(in_channels, n_convs, filters, kernel_size, activation, pool_size, pool_stride):
  '''
  Defines a block in the VGG network.

  Args:
    in_channels (int) -- number of channels of the block's input
    n_convs (int) -- number of convolution layers to append
    filters (int) -- number of filters for the convolution layers
    kernel_size (int or tuple) -- kernel size of the convolution layers
    activation (nn.Module class) -- activation to use after each convolution
    pool_size (int) -- size of the pooling layer
    pool_stride (int) -- stride of the pooling layer

  Returns:
    nn.Sequential containing the convolutions followed by the max pooling layer
  '''
  layers = []
  for i in range(n_convs):
      layers.append(nn.Conv2d(in_channels, filters, kernel_size=kernel_size, padding='same'))
      layers.append(activation())
      in_channels = filters

  layers.append(nn.MaxPool2d(kernel_size=pool_size, stride=pool_stride))

  return nn.Sequential(*layers)

### Pre-trained VGG weights

In the Keras version you downloaded a `.h5` file with the ImageNet weights of VGG-16 (without the top layers). torchvision ships the same pretrained network, `torchvision.models.vgg16`, and downloads its weights automatically. The helper below copies those weights, convolution by convolution, into the encoder you are about to build.

In [ ]:
def load_vgg16_pretrained_weights(encoder):
  '''
  Copies the ImageNet weights of torchvision's VGG-16 into the 13 convolutions of `encoder`.

  Args:
    encoder (nn.Module) -- the VGG_16 encoder whose convolutions get the weights
  '''
  pretrained = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
  pretrained_convs = [m for m in pretrained if isinstance(m, nn.Conv2d)]
  encoder_convs = [m for m in encoder.modules() if isinstance(m, nn.Conv2d)]
  assert len(pretrained_convs) == len(encoder_convs), "the encoder should have exactly 13 convolutions"

  with torch.no_grad():
    for src, dst in zip(pretrained_convs, encoder_convs):
      dst.weight.copy_(src.weight)
      dst.bias.copy_(src.bias)

### Define VGG-16

You can build the encoder as shown below.

* You will create 5 blocks with increasing number of filters at each stage.
* The number of convolutions, filters, kernel size, activation, pool size and pool stride will remain constant.
* You will load the pretrained weights after creating the VGG 16 network.
* Additional convolution layers will be appended to extract more features.
* The output will contain the output of the last layer and the previous four convolution blocks.

In [ ]:
class VGG_16(nn.Module):
  '''
  This module defines the VGG encoder.

  Its forward pass takes a batch of images and returns a tuple of tensors:
  the output of all encoder blocks plus the final convolution layer
  '''

  def __init__(self):
    '''
    Builds the five VGG blocks, loads the pretrained weights, then appends conv6 and conv7.
    '''
    super().__init__()

    # create 5 blocks with increasing filters at each stage.
    # you will save the output of each block (i.e. p1, p2, p3, p4, p5). "p" stands for the pooling layer.
    self.block1 = block(3,   n_convs=2, filters=64,  kernel_size=(3,3), activation=nn.ReLU, pool_size=(2,2), pool_stride=(2,2))
    self.block2 = block(64,  n_convs=2, filters=128, kernel_size=(3,3), activation=nn.ReLU, pool_size=(2,2), pool_stride=(2,2))
    self.block3 = block(128, n_convs=3, filters=256, kernel_size=(3,3), activation=nn.ReLU, pool_size=(2,2), pool_stride=(2,2))
    self.block4 = block(256, n_convs=3, filters=512, kernel_size=(3,3), activation=nn.ReLU, pool_size=(2,2), pool_stride=(2,2))
    self.block5 = block(512, n_convs=3, filters=512, kernel_size=(3,3), activation=nn.ReLU, pool_size=(2,2), pool_stride=(2,2))

    # load the pretrained weights
    load_vgg16_pretrained_weights(self)

    # number of filters for the output convolutional layers
    n = 4096

    # our input images are 224x224 pixels so they will be downsampled to 7x7 after the pooling layers above.
    # we can extract more features by chaining two more convolution layers.
    self.conv6 = nn.Sequential(nn.Conv2d(512, n, kernel_size=(7, 7), padding='same'), nn.ReLU())
    self.conv7 = nn.Sequential(nn.Conv2d(n, n, kernel_size=(1, 1), padding='same'), nn.ReLU())

  def forward(self, image_input):
    '''
    Runs the image through the five VGG blocks and the two extra convolutions.

    Args:
      image_input (tensor) -- batch of images, shape (N, 3, 224, 224)

    Returns:
      tuple of tensors -- the pool1 to pool4 outputs plus conv7, the last of shape (N, 4096, 7, 7)
    '''
    p1 = self.block1(image_input)
    p2 = self.block2(p1)
    p3 = self.block3(p2)
    p4 = self.block4(p3)
    p5 = self.block5(p4)

    c6 = self.conv6(p5)
    c7 = self.conv7(c6)

    # return the outputs at each stage. you will only need two of these in this particular exercise
    # but we included it all in case you want to experiment with other types of decoders.
    return (p1, p2, p3, p4, c7)

### Define FCN 8 Decoder

Next, you will build the decoder using deconvolution layers. Please refer to the diagram for FCN-8 at the start of this section to visualize what the code below is doing. It will involve two summations before upsampling to the original image size and generating the predicted mask.

- Keras' `Conv2DTranspose` is `nn.ConvTranspose2d`, and `Cropping2D(cropping=(1,1))` is simply slicing one pixel off each border.
- The Keras decoder ended with a `softmax` activation. Here the decoder returns the raw class scores (logits) because `nn.CrossEntropyLoss` applies the softmax itself; you'll apply `softmax`/`argmax` explicitly when you make predictions.

In [ ]:
class FCN8Decoder(nn.Module):
  '''
  Defines the FCN 8 decoder.

  Args:
    n_classes (int) - number of classes

  The forward pass takes the tuple of encoder outputs and returns a tensor with shape
  (batch, n_classes, height, width) containing the class scores (logits) for every pixel.
  '''

  def __init__(self, n_classes):
    '''
    Builds the transposed convolutions and the 1x1 convolutions for the skips.

    Args:
      n_classes (int) -- number of segmentation classes
    '''
    super().__init__()

    # upsample the output of the encoder (2x)
    self.upsample_1 = nn.ConvTranspose2d(4096, n_classes, kernel_size=(4,4), stride=(2,2), bias=False)
    # 1x1 convolution that turns the pool 4 features into class predictions
    self.pool4_conv = nn.Sequential(nn.Conv2d(512, n_classes, kernel_size=(1,1), padding='same'), nn.ReLU())

    # upsample the result (2x)
    self.upsample_2 = nn.ConvTranspose2d(n_classes, n_classes, kernel_size=(4,4), stride=(2,2), bias=False)
    # 1x1 convolution that turns the pool 3 features into class predictions
    self.pool3_conv = nn.Sequential(nn.Conv2d(256, n_classes, kernel_size=(1,1), padding='same'), nn.ReLU())

    # upsample up to the size of the original image (8x)
    self.upsample_3 = nn.ConvTranspose2d(n_classes, n_classes, kernel_size=(8,8), stride=(8,8), bias=False)

  def forward(self, convs):
    '''
    Upsamples in three stages, adding the pool 4 and pool 3 predictions.

    Args:
      convs (tuple) -- the encoder outputs (f1, f2, f3, f4, f5)

    Returns:
      tensor -- per-pixel class scores, shape (N, n_classes, 224, 224)
    '''
    # unpack the output of the encoder
    f1, f2, f3, f4, f5 = convs

    # upsample the output of the encoder then crop extra pixels that were introduced
    o = self.upsample_1(f5)
    o = o[:, :, 1:-1, 1:-1]

    # load the pool 4 prediction and do a 1x1 convolution to reshape it to the same shape of `o` above
    o2 = f4
    o2 = self.pool4_conv(o2)

    # add the results of the upsampling and pool 4 prediction
    o = o + o2

    # upsample the resulting tensor of the operation you just did
    o = self.upsample_2(o)
    o = o[:, :, 1:-1, 1:-1]

    # load the pool 3 prediction and do a 1x1 convolution to reshape it to the same shape of `o` above
    o2 = f3
    o2 = self.pool3_conv(o2)

    # add the results of the upsampling and pool 3 prediction
    o = o + o2

    # upsample up to the size of the original image
    o = self.upsample_3(o)

    return o

### Define Final Model

You can now build the final model by connecting the encoder and decoder blocks.

In [ ]:
class SegmentationModel(nn.Module):
  '''
  Defines the final segmentation model by chaining together the encoder and decoder.
  '''

  def __init__(self, n_classes=12):
    '''
    Builds the VGG encoder and the FCN-8 decoder.

    Args:
      n_classes (int) -- number of segmentation classes
    '''
    super().__init__()
    self.encoder = VGG_16()
    self.decoder = FCN8Decoder(n_classes)

  def forward(self, inputs):
    '''
    Runs the image through the encoder and then the decoder.

    Args:
      inputs (tensor) -- batch of images, shape (N, 3, 224, 224)

    Returns:
      tensor -- per-pixel class scores, shape (N, n_classes, 224, 224)
    '''
    convs = self.encoder(inputs)
    outputs = self.decoder(convs)
    return outputs


def segmentation_model(device):
  '''
  Creates the segmentation model on the chosen device.

  Args:
    device (torch.device) -- device the model is moved to

  Returns:
    nn.Module -- the encoder and decoder chained together
  '''
  return SegmentationModel(12).to(device)

In [ ]:
# instantiate the model and see how it looks
model = segmentation_model(device)
summary(model, input_size=(1, 3, 224, 224), device=device, depth=3)

### Configure the Model for Training

Next, the training will be configured. You will need to specify the loss, optimizer and metrics. You will use `nn.CrossEntropyLoss` as the loss function: it is the equivalent of Keras' `categorical_crossentropy`, computed for every pixel from the integer class ids in the label map.

In [ ]:
sgd = torch.optim.SGD(model.parameters(), lr=1E-2, momentum=0.9, nesterov=True)

loss_fn = nn.CrossEntropyLoss()

## Train the Model

The model can now be trained. This will take a while to run (the original lab quotes around 30 minutes on a GPU) and you will reach around 85% accuracy for both train and val sets.

In [ ]:
EPOCHS = 170

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}


def run_epoch(loader, model, loss_fn, optimizer, device, train):
  '''
  Runs one pass over `loader`, training or evaluating.

  Args:
    loader (DataLoader) -- yields (images, label maps) batches
    model (nn.Module) -- the segmentation model
    loss_fn (callable) -- loss applied to (logits, label maps)
    optimizer (Optimizer) -- updates weights; only used when train is True
    device (torch.device) -- device the batches are moved to
    train (bool) -- True updates the weights, False only measures

  Returns:
    (float, float) -- mean loss and mean per-pixel accuracy
  '''
  model.train(train)
  total_loss, correct, count = 0.0, 0, 0
  with torch.set_grad_enabled(train):
    for images, annotations in loader:
      images, annotations = images.to(device), annotations.to(device)
      logits = model(images)
      loss = loss_fn(logits, annotations)
      if train:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
      total_loss += loss.item() * len(images)
      correct += (logits.argmax(1) == annotations).float().mean().item() * len(images)
      count += len(images)
  return total_loss / count, correct / count


for epoch in range(EPOCHS):
  train_loss, train_acc = run_epoch(training_dataset, model, loss_fn, sgd, device, train=True)
  val_loss, val_acc = run_epoch(validation_dataset, model, loss_fn, sgd, device, train=False)
  history['loss'].append(train_loss); history['accuracy'].append(train_acc)
  history['val_loss'].append(val_loss); history['val_accuracy'].append(val_acc)
  print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

## Evaluate the Model

After training, you will want to see how your model is doing on a test set. For segmentation models, you can use the intersection-over-union and the dice score as metrics to evaluate your model. You'll see how it is implemented in this section.

In [ ]:
def get_images_and_segments_test_arrays(validation_dataset):
  '''
  Gets a subsample of the val set as your test set.

  Args:
    validation_dataset (DataLoader) -- batches of the validation split

  Returns:
    (tensor, array) -- the ground truth images and their label maps
  '''
  test_count = 64

  # the first batch of the validation loader holds the first 64 images and their label maps
  y_true_images, y_true_segments = next(iter(validation_dataset))

  y_true_images = y_true_images[:test_count]
  y_true_segments = y_true_segments[:test_count].numpy()

  return y_true_images, y_true_segments

# load the ground truth images and segmentation masks
y_true_images, y_true_segments = get_images_and_segments_test_arrays(validation_dataset)

### Make Predictions

You can get output segmentation masks by running the model in evaluation mode. The output of our segmentation model has the shape `(batch, 12, height, width)` where `12` is the number of classes. After a `softmax` over that axis, each value indicates the probability of that pixel belonging to that particular class. If you want to create the predicted label map, then you can get the `argmax()` of that axis. This is shown in the following cell.

In [ ]:
# get the model prediction
model.eval()
with torch.no_grad():
  results = torch.softmax(model(y_true_images.to(device)), dim=1).cpu().numpy()

# for each pixel, get the slice number which has the highest probability
results = np.argmax(results, axis=1)

### Compute Metrics

The function below generates the IOU and dice score of the prediction and ground truth masks. From the lectures, it is given that:

$$IOU = \frac{area\_of\_overlap}{area\_of\_union}$$
<br>
$$Dice Score = 2 * \frac{area\_of\_overlap}{combined\_area}$$

The code below does that for you. A small smoothening factor is introduced in the denominators to prevent possible division by zero.

In [ ]:
def compute_metrics(y_true, y_pred):
  '''
  Computes IOU and Dice Score.

  Args:
    y_true (array) -- ground truth label map
    y_pred (array) -- predicted label map

  Returns:
    (list, list) -- IOU and Dice score, one entry per class
  '''

  class_wise_iou = []
  class_wise_dice_score = []

  smoothening_factor = 0.00001

  for i in range(12):
    intersection = np.sum((y_pred == i) * (y_true == i))
    y_true_area = np.sum((y_true == i))
    y_pred_area = np.sum((y_pred == i))
    combined_area = y_true_area + y_pred_area

    iou = (intersection + smoothening_factor) / (combined_area - intersection + smoothening_factor)
    class_wise_iou.append(iou)

    dice_score =  2 * ((intersection + smoothening_factor) / (combined_area + smoothening_factor))
    class_wise_dice_score.append(dice_score)

  return class_wise_iou, class_wise_dice_score

### Show Predictions and Metrics

You can now see the predicted segmentation masks side by side with the ground truth. The metrics are also overlayed so you can evaluate how your model is doing.

In [ ]:
# input a number from 0 to 63 to pick an image from the test set
integer_slider = 0

# compute metrics
iou, dice_score = compute_metrics(y_true_segments[integer_slider], results[integer_slider])

# visualize the output and metrics
show_predictions(y_true_images[integer_slider], [results[integer_slider], y_true_segments[integer_slider]], ["Image", "Predicted Mask", "True Mask"], iou, dice_score)

### Display Class Wise Metrics

You can also compute the class-wise metrics so you can see how your model performs across all images in the test set.

In [ ]:
# compute class-wise metrics
cls_wise_iou, cls_wise_dice_score = compute_metrics(y_true_segments, results)

In [ ]:
# print IOU for each class
for idx, iou in enumerate(cls_wise_iou):
  spaces = ' ' * (13-len(class_names[idx]) + 2)
  print("{}{}{} ".format(class_names[idx], spaces, iou))

In [ ]:
# print the dice score for each class
for idx, dice_score in enumerate(cls_wise_dice_score):
  spaces = ' ' * (13-len(class_names[idx]) + 2)
  print("{}{}{} ".format(class_names[idx], spaces, dice_score))

**That's all for this lab! In the next section, you will work on another architecture for building a segmentation model: the UNET.**